In [1]:
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Temurin-17.0.19+10 (build 17.0.19+10)
OpenJDK 64-Bit Server VM Temurin-17.0.19+10 (build 17.0.19+10, mixed mode)


# Template Quest Notebook

In [26]:
# Import neccessary modules, add to this cell as needed
# Provided PySpark examples
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F

## Part 1: Load the Sample Dataset

### Note
I thought I'd leave in working steps to show process but I've commented out **print**s and **show**s to save clutter 

In [27]:
# Initiate a new Spark session and set the case sensitivity option
spark = (
    SparkSession.builder
        .appName("cyberquest")
        .getOrCreate()
)
spark.conf.set("spark.sql.caseSensitive", True)

In [38]:
# TODO: Load the raw data
#    - read from relative path
data_file = 'data/sysmon_spearphish_cribl.json'
df_bronze = spark.read.json(data_file)
df_bronze.show(10, truncate = False)

+----------------------------------------------+-----------+---------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [44]:
#local copy of first three rows for a bit of manual exploration.
inspectionSample = df_bronze.take(3)
print(inspectionSample[2])


Row(Computer='win-host-ctus-attack-range-212', Description=None, EventCode='23', User='NT AUTHORITY\\SYSTEM', _raw='{"Name":"\'Microsoft-Windows-Sysmon\'","Guid":"\'{5770385F-C22A-43E0-BF4C-06F5698FFBD9}\'","Version":"5","Level":"4","Task":"23","Opcode":"0","Keywords":"0x8000000000000000","SystemTime":"\'2023-01-27T11:19:47.009197600Z\'","EventRecordID":"320270","ProcessID":"\'1912\'","ThreadID":"\'2628\'","Channel":"Microsoft-Windows-Sysmon/Operational","UserID":"S-1-5-18","RuleName":"-","UtcTime":"2023-01-27 11:19:47.008","ProcessGuid":"{72106695-9B85-63D3-1B00-00000000BD02}","ProcessId":"1880","Image":"C:\\\\Program Files\\\\Amazon\\\\SSM\\\\amazon-ssm-agent.exe","TargetFilename":"C:\\\\ProgramData\\\\Amazon\\\\SSM\\\\InstanceData\\\\i-0f154ddf1aa5a8e8d\\\\channels\\\\health\\\\surveyor-20230127093814-099","Hashes":"MD5=97EF2A570B75C4F95FC69B0D09A2E2A2,SHA256=11396EA313B0ED7E3228C4FA92ABE9D836DB8F416A7A8A28ACC77133025082E7,IMPHASH=00000000000000000000000000000000","IsExecutable":"fa

In [45]:
# TODO: Parse the raw data into something relevant and usable
#   - make fields in _raw usable, flatten into 2D df. 
#      - [x] make rough schema of inner JSON, just useful looking fields.
#          - [x] if task 2 requires further fields or I miss any I can revisit here.
#      - [x] make copy of bronze_df with another column containing _raw as parsed json from rough schema.
#      - [x] select useful fields from parsed_raw as columns for silver_df.

rough_schema = StructType([
    StructField("UtcTime", StringType(), True),
    StructField("Image", StringType(), True),
    StructField("ProcessId", StringType(), True),
    StructField("CommandLine", StringType(), True),
    StructField("ParentImage", StringType(), True),
    StructField("TargetFilename", StringType(), True),
    StructField("SourceImage", StringType(), True),
    StructField("TargetImage", StringType(), True),
    StructField("QueryName", StringType(), True),
    StructField("Hashes", StringType(), True),
])

df_bronze_copy = df_bronze.withColumn(
    "raw_parsed",
    from_json(col("_raw"), rough_schema)
)

In [48]:
df_bronze_copy.show(5)

+--------------------+-----------+---------+-------------------+--------------------+----------------+-------------+--------------------+------------+--------------------+--------------------+
|            Computer|Description|EventCode|               User|                _raw|           _time|cribl_breaker|          cribl_pipe|        host|              source|          raw_parsed|
+--------------------+-----------+---------+-------------------+--------------------+----------------+-------------+--------------------+------------+--------------------+--------------------+
|win-host-ctus-att...|       NULL|       23|NT AUTHORITY\SYSTEM|{"Name":"'Microso...|1.674818387593E9|     fallback|Splunk_UF_Windows...|f2c75b47cbb7|/tmp/cribldata/in...|{2023-01-27 11:19...|
|win-dc-ctus-attac...|       NULL|       23|NT AUTHORITY\SYSTEM|{"Name":"'Microso...|1.674818387419E9|     fallback|Splunk_UF_Windows...|f2c75b47cbb7|/tmp/cribldata/in...|{2023-01-27 11:19...|
|win-host-ctus-att...|       NULL| 

In [50]:
df_silver = df_bronze_copy.select(
    #outer
    col("Computer"),
    col("EventCode"),
    col("User"),
    col("_time"),
    col("source"),
    #inner
    col("raw_parsed.UtcTime").alias("UtcTime"),
    col("raw_parsed.Image").alias("Image"),
    col("raw_parsed.ProcessId").alias("ProcessId"),
    col("raw_parsed.CommandLine").alias("CommandLine"),
    col("raw_parsed.ParentImage").alias("ParentImage"),
    col("raw_parsed.TargetFilename").alias("TargetFilename"),
    col("raw_parsed.QueryName").alias("QueryName"),
    col("raw_parsed.SourceImage").alias("SourceImage"),
    col("raw_parsed.TargetImage").alias("TargetImage"),
    col("raw_parsed.Hashes").alias("Hashes")
    
)

In [51]:
#df_flat.show(5)

In [52]:
df_silver.createOrReplaceTempView("sysmon_silver")

In [53]:
# Provided PySpark SQL Example
spark.sql("""
SELECT *
FROM sysmon_silver
LIMIT 5
""").show()

+--------------------+---------+-------------------+----------------+--------------------+--------------------+--------------------+---------+-----------+-----------+--------------------+---------+--------------------+--------------------+--------------------+
|            Computer|EventCode|               User|           _time|              source|             UtcTime|               Image|ProcessId|CommandLine|ParentImage|      TargetFilename|QueryName|         SourceImage|         TargetImage|              Hashes|
+--------------------+---------+-------------------+----------------+--------------------+--------------------+--------------------+---------+-----------+-----------+--------------------+---------+--------------------+--------------------+--------------------+
|win-host-ctus-att...|       23|NT AUTHORITY\SYSTEM|1.674818387593E9|/tmp/cribldata/in...|2023-01-27 11:19:...|C:\Program Files\...|     3764|       NULL|       NULL|C:\Program Files\...|     NULL|                NULL

## Part 2: Detection Engineering

### Hypothesis
- *Adversaries may send spearphishing emails with a malicious attachment in an attempt to gain access to victim systems.*
- *A threat actor can leverage Microsoft Office applications against users on Windows endpoints through phishing to make potentially malicious web calls.*
- ***Windows Endpoints forwarding DNS activity via sysmon logs** to a data orchestration platform like Cribl would be utilized to identify this activity.*

Preferably using either PySpark or SQL, create a query or queries that will prove the detection hypothesis.
- [ ] Output the result of this query in the notebook with a new dataframe
- [ ] Select columns based on what is applicable to an analyst during triage



### Notes


-
- fields
- no sender field
- no domain field
- identify windows endpoint forwarding DNS activity via sysmon logs to a 'crible'
   - sysmon eventID 22
      - *'DNS Queries made by any process on a Windows machine'*
- look for external DNS connection from newly started process
- 
- sysmon process start 
- 

In [74]:
# DNS sysmon Events
spark.sql("""
SELECT Computer, EventCode, source, Image, QueryName
FROM sysmon_silver
WHERE 
    EventCode = 22
""").show(truncate = False)

+------------------------------+---------+---------------------------------------------------+------------------------------------------------------------------------------+---------------------------------+
|Computer                      |EventCode|source                                             |Image                                                                         |QueryName                        |
+------------------------------+---------+---------------------------------------------------+------------------------------------------------------------------------------+---------------------------------+
|win-host-ctus-attack-range-212|22       |/tmp/cribldata/in/sysmon_office_doc_abuses_rels.log|C:\Temp\OfficeSetup.exe                                                       |ecs.office.com                   |
|win-host-ctus-attack-range-212|22       |/tmp/cribldata/in/sysmon_office_doc_abuses_rels.log|C:\Windows\System32\svchost.exe                                           

In [76]:
# excluding expected MS destinations
spark.sql("""
SELECT Computer, EventCode, source, Image, QueryName
FROM sysmon_silver
WHERE 
    EventCode = 22
    AND QueryName NOT LIKE '%.office.%'
    AND QueryName NOT LIKE '%microsoft%'
""").show(truncate = False)

+------------------------------+---------+---------------------------------------------------+-----------------------------------------------------------+-----------------+
|Computer                      |EventCode|source                                             |Image                                                      |QueryName        |
+------------------------------+---------+---------------------------------------------------+-----------------------------------------------------------+-----------------+
|win-host-ctus-attack-range-212|22       |/tmp/cribldata/in/sysmon_office_doc_abuses_rels.log|C:\Windows\System32\svchost.exe                            |oneclient.sfx.ms |
|win-host-ctus-attack-range-212|22       |/tmp/cribldata/in/sysmon_office_doc_abuses_rels.log|C:\Program Files\Microsoft Office\root\Office16\WINWORD.EXE|www.mediafire.com|
+------------------------------+---------+---------------------------------------------------+-----------------------------------------

In [69]:
# all sources are cribl
spark.sql("""
SELECT *
FROM sysmon_silver
WHERE 
    source NOT LIKE '%cribl%'
""").show(truncate = False)

+--------+---------+----+-----+------+-------+-----+---------+-----------+-----------+--------------+---------+-----------+-----------+------+
|Computer|EventCode|User|_time|source|UtcTime|Image|ProcessId|CommandLine|ParentImage|TargetFilename|QueryName|SourceImage|TargetImage|Hashes|
+--------+---------+----+-----+------+-------+-----+---------+-----------+-----------+--------------+---------+-----------+-----------+------+
+--------+---------+----+-----+------+-------+-----+---------+-----------+-----------+--------------+---------+-----------+-----------+------+



## Part 3: Additional Steps

### Part 3.1: Normalization

### Part 3.2: Alert Table

### Part 3.3: Enrichment

## Summary

Summarize your submission here, comments are helpful to add throughout your code as well.